In [32]:
import cvxpy as cp
import gurobipy as gp
import heapq
import itertools
import numpy as np
import pandas as pd
import time
import tqdm
import warnings

from gurobipy import GRB
warnings.filterwarnings("ignore")

In [33]:
def validate(priors, thresholds, tt, c):
    p = np.asarray(priors, dtype=float)
    t = np.asarray(thresholds, dtype=float)
    if p.shape != t.shape or p.ndim != 1:
        raise ValueError("priors and thresholds must be 1-D of equal length")
    if np.any(p < 0):
        raise ValueError("priors must be nonnegative")
    if not np.isclose(p.sum(), 1.0):
        raise ValueError(f"priors must sum 1, got {p.sum()}")
    if c <= 0:
        raise ValueError("c must be positive")
    return p, t, float(tt), float(c)
 
 
def build_grid(m, lo=0.0, hi=1.0):
    X = np.linspace(lo, hi, m)
    D = np.full(m, 1.0 / m)
    return X, D
 
 
def row(x_idx, j, n):
    return x_idx * n + j
 
 
def assignment_to_partition(assign):
    bins = {}
    for i, j in enumerate(assign):
        bins.setdefault(j, []).append(i)
    return frozenset(frozenset(v) for v in bins.values())
 
 
def add_symmetry(K, xv):
    n = K["n"]
    cons = [xv[i, j] == 0 for i in range(n) for j in range(i + 1, n)]
    for i in range(1, n):
        for j in range(1, i + 1):
            cons.append(xv[i, j] <= cp.sum(xv[:i, j - 1]))
    return cons
 
 
def pin_empty_bins(K, xv, yv):
    n = K["n"]
    occ = cp.sum(xv, axis=0)
    return [yv[x_idx * n:(x_idx + 1) * n, 0] >= 1 - occ for x_idx in range(K["m"])]
 
 
def add_objective(K, xv, yv):
    n, m, L, D, p = K["n"], K["m"], K["L"], K["D"], K["p"]
    v = cp.Variable((m * n, n), nonneg=True, name="v")
    cons = []
    for x_idx in range(m):
        sl = slice(x_idx * n, (x_idx + 1) * n)
        g = yv[sl, :] @ L[x_idx].T
        cons.append(v[sl, :] >= g + xv.T - 1)
    W = np.repeat(D, n)[:, None] * p[None, :]
    return cp.sum(cp.multiply(W, v)), cons
 
 
def evaluate_partition(K, assign):
    n, m, U, L, D, p, valid = (K["n"], K["m"], K["U"], K["L"], K["D"],
                               K["p"], K["valid"])
    total = 0.0
    for j in range(n):
        members = np.flatnonzero(assign == j)
        if members.size == 0:
            continue
        B = np.zeros(n)
        B[members] = 1.0
        score = (U * B[None, :, None]).sum(axis=1)
        a_star = np.where(valid, score, -np.inf).argmax(axis=1)
        err = L[np.arange(m)[:, None], members[None, :], a_star[:, None]]
        total += float(D @ (err @ p[members]))
    return total


def build_constants(priors, thresholds, tt, c, m, eps=1e-6, tight_M=False):
    priors, thresholds, tt, c = validate(priors, thresholds, tt, c)
    n = len(thresholds)
    X, D = build_grid(m)
 
    A = np.empty((m, n + 1))
    A[:, 0] = X
    A[:, 1:] = thresholds[None, :]
 
    cost = c * np.abs(A - X[:, None])
    Hx = (A[:, None, :] >= thresholds[None, :, None]).astype(float)
    U = priors[None, :, None] * (Hx - (1.0 + eps) * cost[:, None, :])
    f = (X >= tt).astype(float)
    L = np.where(f[:, None, None] == 0.0, Hx, 1.0 - Hx)
    M = 1.0 + (1.0 + eps) * cost.max(axis=1)
 
    valid = A >= X[:, None] - 1e-12
    for xi in range(m):
        seen = set()
        for a in range(n + 1):
            key = round(float(A[xi, a]), 12)
            if key in seen:
                valid[xi, a] = False
            else:
                seen.add(key)
 
    # The BR row holds a SUBSET sum selected by x[:, j], so the valid bound is
    # the max over subsets -- the sum of positive parts -- not a bound on the
    # full sum over all i. M3[x,a,a'] is that value, and it is the tightest
    # constant that keeps every partition feasible.
    diff = U.transpose(0, 2, 1)                      # (m, a, i)
    M3 = np.maximum(diff[:, None, :, :] - diff[:, :, None, :], 0.0).sum(axis=3)
    # M3[x, a, a'] = sum_i (U[x,i,a'] - U[x,i,a])_+
 
    return dict(p=priors, t=thresholds, tt=tt, c=c, n=n, m=m, X=X, D=D, A=A,
                cost=cost, Hx=Hx, U=U, L=L, M=M, M3=M3, eps=eps, valid=valid,
                tight_M=tight_M)

In [34]:
def build_variables(K, relax=False):
    n, m = K["n"], K["m"]
    if relax:
        xv = cp.Variable((n, n), name="x")
        yv = cp.Variable((m * n, n + 1), name="y")
        cons = [xv >= 0, xv <= 1, yv >= 0, yv <= 1,
                cp.sum(xv, axis=1) == 1,
                cp.sum(yv, axis=1) == 1]
    else:
        xv = cp.Variable((n, n), boolean=True, name="x")
        yv = cp.Variable((m * n, n + 1), boolean=True, name="y")
        cons = [cp.sum(xv, axis=1) == 1,
                cp.sum(yv, axis=1) == 1]
    return xv, yv, cons


def add_best_response(K, xv, yv, tight_M=None):
    n, m, U, M, M3, valid = K["n"], K["m"], K["U"], K["M"], K["M3"], K["valid"]
    if tight_M is None:
        tight_M = K.get("tight_M", False)
    ones = np.ones((1, n + 1))
    cons = []
 
    for x_idx in range(m):
        util = xv.T @ U[x_idx]
        y_x = yv[row(x_idx, 0, n): row(x_idx, 0, n) + n, :]
 
        for a in range(n + 1):
            if not valid[x_idx, a]:
                cons.append(y_x[:, a] == 0)
                continue
            u_a = cp.reshape(util[:, a], (n, 1), order="C")
            y_a = cp.reshape(y_x[:, a], (n, 1), order="C")
            if tight_M:
                slack = (1 - y_a) @ M3[x_idx, a][None, :]      # (n, n+1)
                cons.append(u_a @ ones + slack >= util)
            else:
                cons.append((u_a + M[x_idx] * (1 - y_a)) @ ones >= util)
    return cons


def build_problem(K, relax, reduce=True, pin_empty=True, tight_M=None):
    xv, yv, cons = build_variables(K, relax=relax)
    cons = cons + add_best_response(K, xv, yv, tight_M=tight_M)
    if reduce:
        cons = cons + add_symmetry(K, xv)
    if pin_empty:
        cons = cons + pin_empty_bins(K, xv, yv)
    obj, obj_cons = add_objective(K, xv, yv)
    return cp.Problem(cp.Minimize(obj), cons + obj_cons), xv, yv


def _solver_kwargs(solver, feas_tol, time_limit, relax):
    name = str(solver).upper()
    if "GUROBI" in name:
        kw = {"FeasibilityTol": feas_tol}
        if not relax:
            kw["IntFeasTol"] = feas_tol
            kw["MIPGap"] = 0.0
        if time_limit is not None:
            kw["TimeLimit"] = time_limit
    else:
        kw = dict(primal_feasibility_tolerance=feas_tol)
        if not relax:
            kw["mip_feasibility_tolerance"] = feas_tol
            kw["mip_rel_gap"] = 0.0
        if time_limit is not None:
            kw["time_limit"] = time_limit
    return kw


def solve_one(K, relax, reduce=True, pin_empty=True, tight_M=None, feas_tol=1e-9, time_limit=None, solver=cp.GUROBI):
    """Solve the problem once, as an LP (relax=True) or an IP (relax=False)."""
    prob, xv, yv = build_problem(K, relax, reduce, pin_empty, tight_M)
    kw = _solver_kwargs(solver, feas_tol, time_limit, relax)
    t0 = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        prob.solve(solver=solver, **kw)
    elapsed = time.perf_counter() - t0
 
    out = dict(status=prob.status, seconds=elapsed,
               objective=None if prob.value is None else float(prob.value),
               n_vars=sum(v.size for v in prob.variables()),
               n_cons=len(prob.constraints), relaxed=relax)
 
    if prob.status in ("optimal", "optimal_inaccurate") and not relax:
        assign = np.asarray(xv.value).argmax(axis=1)
        out["assign"] = assign
        out["partition"] = sorted(sorted(b)
                                  for b in assignment_to_partition(assign))
        # exact loss of the partition actually returned; if the solver objective
        # sits below this, the tie-break eps was inert
        out["loss"] = evaluate_partition(K, assign)
    if relax and yv.value is not None:
        Y = np.asarray(yv.value).reshape(K["m"], K["n"], K["n"] + 1)
        frac = np.minimum(Y, 1 - Y)
        out["y_fractionality"] = float(frac.max())
        out["y_frac_mean"] = float(frac.mean())
        Xv = np.asarray(xv.value)
        out["x_fractionality"] = float(np.minimum(Xv, 1 - Xv).max())
    return out


def integrality_gap(priors, thresholds, tt, c, m=21, eps=1e-6, reduce=True, pin_empty=True, tight_M=False, feas_tol=1e-9, time_limit=None, solver=cp.GUROBI):
    """LP relaxation vs IP on the same formulation.
 
    gap_abs   = OPT_IP - OPT_LP
    gap_rel   = (OPT_IP - OPT_LP) / OPT_IP
    tightness = OPT_LP / OPT_IP   in [0,1]; 1 means the relaxation is exact
 
    The IP value reported is the EXACT loss of the returned partition
    (evaluate_partition), not the solver objective, so an inert tie-break eps
    shows up as certified=False rather than as a silently low number.
    """
    K = build_constants(priors, thresholds, tt, c, m, eps, tight_M)
    ip = solve_one(K, False, reduce, pin_empty, tight_M, feas_tol, time_limit, solver)
    lp = solve_one(K, True, reduce, pin_empty, tight_M, feas_tol, time_limit, solver)
 
    ip_val = ip.get("loss")
    lp_val = lp.get("objective")
    ok = ip_val is not None and lp_val is not None and ip_val > 0
 
    return dict(
        lp=lp_val, 
        ip=ip_val,
        solver_objective_ip=ip.get("objective"),
        gap_abs=None if not ok else ip_val - lp_val,
        gap_rel=None if not ok else (ip_val - lp_val) / ip_val,
        tightness=None if not ok else lp_val / ip_val,
        partition_ip=ip.get("partition"),
        y_fractionality_lp=lp.get("y_fractionality"),
        x_fractionality_lp=lp.get("x_fractionality"),
        time_lp=lp["seconds"], time_ip=ip["seconds"],
        n_vars_ip=ip["n_vars"], n_cons_ip=ip["n_cons"],
        status_lp=lp["status"], status_ip=ip["status"],
        reduce=reduce, pin_empty=pin_empty, tight_M=tight_M, eps=eps, m=m,
    )

In [35]:
def generate_prior_grid(n_components, n_balls=50):
    step = 1.0 / n_balls
    grids = []
    for combo in itertools.combinations_with_replacement(range(n_components), n_balls - n_components):
        counts = np.bincount(combo, minlength=n_components) + 1
        grids.append((counts * step).round(4))
    return grids

In [ ]:
rng = np.random.default_rng(0)
M = [11]
N = [4]
# M = [21, 31, 41, 51]
# N = [6, 8, 10, 12, 14]
tt = 0.1
c = 1.0

for n in N:
    thresholds = np.linspace(0.0, 1.0, n)
    prior_grid = np.asarray(generate_prior_grid(n, 20))
    prior_grid = prior_grid[rng.choice(np.arange(len(prior_grid)), 200, replace=False)]
    for m in M:
        results = []
        count = 0
        for i, priors in tqdm.tqdm(enumerate(prior_grid), desc=f"[m={m}] [n={n}]", total=len(prior_grid)):
            if count == 100:
                break
            g = integrality_gap(priors, thresholds, tt, c, m=m, tight_M=False)
            g.update({
                "i": i,
                "priors": priors,
                "thresholds": thresholds,
                "c": c,
                "tt": tt,
            })
            results.append(g)
            if g["status_ip"] == "optimal":
                count += 1
        
        df = pd.DataFrame(results)
        df.to_pickle(f"grid_search_integrality_gap_m{m}_n{n}.pkl")

[m=11] [n=4]:  50%|█████     | 100/200 [00:13<00:13,  7.63it/s]


In [37]:
df

,lp,ip,solver_objective_ip,gap_abs,gap_rel,tightness,partition_ip,y_fractionality_lp,x_fractionality_lp,time_lp,...,reduce,pin_empty,tight_M,eps,m,i,priors,thresholds,c,tt
0,0.013636,0.081818,0.081818,0.068182,0.833333,0.166667,"[[0, 1, 2], [3]]",0.500000,0.500000,0.078154,...,True,True,False,0.000001,11,0,"[0.15, 0.4, 0.35, 0.1]","[0.0, 0.3333333333333333, 0.6666666666666666, ...",1.0,0.1
1,0.009091,0.086364,0.086364,0.077273,0.894737,0.105263,"[[0, 1], [2], [3]]",0.500000,0.500000,0.045953,...,True,True,False,0.000001,11,1,"[0.1, 0.45, 0.4, 0.05]","[0.0, 0.3333333333333333, 0.6666666666666666, ...",1.0,0.1
2,0.031818,0.031818,0.031818,0.000000,0.000000,1.000000,"[[0, 1, 2], [3]]",0.173470,0.173470,0.074233,...,True,True,False,0.000001,11,2,"[0.35, 0.05, 0.55, 0.05]","[0.0, 0.3333333333333333, 0.6666666666666666, ...",1.0,0.1
3,0.022727,0.054545,0.054545,0.031818,0.583333,0.416667,"[[0], [1], [2, 3]]",0.157636,0.157636,0.047034,...,True,True,False,0.000001,11,3,"[0.25, 0.35, 0.05, 0.35]","[0.0, 0.3333333333333333, 0.6666666666666666, ...",1.0,0.1
4,0.045455,0.063636,0.063636,0.018182,0.285714,0.714286,"[[0, 1], [2], [3]]",0.215687,0.215687,0.046591,...,True,True,False,0.000001,11,4,"[0.5, 0.2, 0.2, 0.1]","[0.0, 0.3333333333333333, 0.6666666666666666, ...",1.0,0.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.004545,0.081818,0.081818,0.077273,0.944444,0.055556,"[[0, 1, 2], [3]]",0.500000,0.500000,0.050989,...,True,True,False,0.000001,11,95,"[0.05, 0.3, 0.55, 0.1]","[0.0, 0.3333333333333333, 0.6666666666666666, ...",1.0,0.1
96,0.004545,0.018182,0.018182,0.013636,0.750000,0.250000,"[[0], [1], [2, 3]]",0.500000,0.500000,0.047008,...,True,True,False,0.000001,11,96,"[0.05, 0.15, 0.5, 0.3]","[0.0, 0.3333333333333333, 0.6666666666666666, ...",1.0,0.1
97,0.009091,0.022727,0.022727,0.013636,0.600000,0.400000,"[[0, 1], [2, 3]]",0.500000,0.500000,0.047374,...,True,True,False,0.000001,11,97,"[0.1, 0.15, 0.45, 0.3]","[0.0, 0.3333333333333333, 0.6666666666666666, ...",1.0,0.1
98,0.013636,0.072727,0.072727,0.059091,0.812500,0.187500,"[[0], [1, 2], [3]]",0.500000,0.500000,0.047711,...,True,True,False,0.000001,11,98,"[0.15, 0.1, 0.55, 0.2]","[0.0, 0.3333333333333333, 0.6666666666666666, ...",1.0,0.1


In [27]:
results

[{'lp': 0.013636363636363636,
  'ip': 0.08181818181818182,
  'solver_objective': 0.08181818181818182,
  'certified': True,
  'gap_abs': 0.06818181818181818,
  'gap_rel': 0.8333333333333333,
  'tightness': 0.16666666666666666,
  'partition': [[0, 1, 2], [3]],
  'y_fractionality': 0.49999952272729453,
  'x_fractionality': 0.49999952272729453,
  'lp_seconds': 0.04596816701814532,
  'ip_seconds': 0.07145679206587374,
  'n_vars': 412,
  'n_cons': 91,
  'status_lp': 'optimal',
  'status_ip': 'optimal',
  'reduce': True,
  'pin_empty': True,
  'tight_M': False,
  'eps': 1e-06,
  'm': 11,
  'i': 0,
  'priors': array([0.15, 0.4 , 0.35, 0.1 ]),
  'thresholds': array([0.        , 0.33333333, 0.66666667, 1.        ]),
  'c': 1.0,
  'tt': 0.1},
 {'lp': 0.009090909090909092,
  'ip': 0.08636363636363636,
  'solver_objective': 0.08636363636363636,
  'certified': True,
  'gap_abs': 0.07727272727272727,
  'gap_rel': 0.8947368421052632,
  'tightness': 0.10526315789473685,
  'partition': [[0, 1], [2], [3]

In [7]:
def rand_inst(rng, n):
    t = np.sort(rng.choice(np.arange(1, 20)/20, size=n, replace=False))
    p = rng.integers(1, 10, size=n).astype(float); p /= p.sum()
    tau = float(rng.choice(np.arange(1, 20)/20))
    c = float(rng.choice([0.5, 1., 2., 3.]))
    return p, t, tau, c

In [8]:
print("=== tightness by n (symmetry on, m=11) ===")
print(f"{'n':>3} {'trials':>7} {'loose M':>18} {'tight M':>18}")
for n in [3, 4, 5]:
    rng = np.random.default_rng(100 + n)
    N = 15 if n < 5 else 8
    a, b = [], []
    for _ in range(N):
        p, t, tau, c = rand_inst(rng, n)
        g1 = integrality_gap(p, t, tau, c, m=11, tight_M=False)
        g2 = integrality_gap(p, t, tau, c, m=11, tight_M=True)
        if g1["tightness"] is not None: a.append(g1["tightness"])
        if g2["tightness"] is not None: b.append(g2["tightness"])
        assert g1["certified"], "tie-break eps was inert"
    print(f"{n:>3} {N:>7}   mean {np.mean(a):.4f} max {np.max(a):.4f}"
          f"   mean {np.mean(b):.4f} max {np.max(b):.4f}")

=== tightness by n (symmetry on, m=11) ===
  n  trials            loose M            tight M
  3      15   mean 0.4412 max 0.8504   mean 0.5135 max 0.8905
  4      15   mean 0.3715 max 0.7886   mean 0.4117 max 0.8309
  5       8   mean 0.2170 max 0.7833   mean 0.2666 max 0.8280


In [9]:
print("\n=== does grid resolution change the gap? (n=4, tight M) ===")
rng = np.random.default_rng(7)
p, t, tau, c = rand_inst(rng, 4)
print(f"{'m':>5} {'LP':>12} {'IP':>12} {'tightness':>10} {'ip_sec':>8}")
for m in [6, 11, 21, 41]:
    g = integrality_gap(p, t, tau, c, m=m, tight_M=True)
    print(f"{m:>5} {g['lp']:>12.6f} {g['ip']:>12.6f} {g['tightness']:>10.4f} "
          f"{g['ip_seconds']:>7.2f}s")


=== does grid resolution change the gap? (n=4, tight M) ===
    m           LP           IP  tightness   ip_sec
    6     0.166667     0.166667     1.0000    0.04s
   11     0.143315     0.154545     0.9273    0.06s
   21     0.130597     0.147619     0.8847    0.10s
   41     0.139943     0.175610     0.7969    0.25s


In [10]:
print("\n=== is the LP fractional in x, or only in y? ===")
rng = np.random.default_rng(3)
for _ in range(5):
    p, t, tau, c = rand_inst(rng, 4)
    g = integrality_gap(p, t, tau, c, m=11, tight_M=True)
    print(f"  tightness={g['tightness']:.4f}  max frac in x={g['x_fractionality']:.4f}"
          f"  in y={g['y_fractionality']:.4f}")


=== is the LP fractional in x, or only in y? ===
  tightness=0.8947  max frac in x=0.4800  in y=0.4800
  tightness=0.3207  max frac in x=0.4454  in y=0.4539
  tightness=0.6429  max frac in x=0.4650  in y=0.4567
  tightness=0.7282  max frac in x=0.3078  in y=0.4766
  tightness=0.0943  max frac in x=0.2432  in y=0.4865
